# **Project Description: Next Word Prediction Using LSTM**
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

***1- Data Collection:*** We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

***2- Data Preprocessing:*** The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

***3- Model Building:*** An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

***4- Model Training:*** The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

***5- Model Evaluation:*** The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

***6- Deployment:*** A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

---



In [15]:
# Data Collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [16]:
## load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')
## save to a file
with open('hamlet.txt','w') as file:
  file.write(data)

In [17]:
## Data Preprocessing

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [18]:
## load the dataset
with open('hamlet.txt','r') as file:
  text = file.read().lower()

# Tokenize the text - creating indexes for words
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.index_word)
print("Length of word indexes :" ,total_words)

Length of word indexes : 4817


In [19]:
for index, word in list(tokenizer.index_word.items())[:50]:
  print(f"{index}:{word}")

1:the
2:and
3:to
4:of
5:i
6:you
7:a
8:my
9:it
10:in
11:that
12:ham
13:is
14:not
15:his
16:this
17:with
18:your
19:but
20:for
21:me
22:lord
23:as
24:what
25:he
26:be
27:so
28:him
29:haue
30:king
31:will
32:no
33:our
34:we
35:on
36:are
37:if
38:all
39:then
40:shall
41:by
42:thou
43:come
44:or
45:hamlet
46:good
47:do
48:hor
49:her
50:let


In [20]:
## create input sequence
input_sequences = []
for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

In [21]:
input_sequences[:53]

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [22]:
## pad sequence
max_sequence_len = max([len(x) for x in input_sequences])
max_sequence_len

14

In [23]:
input_sequences = np.array(pad_sequences(input_sequences,maxlen=max_sequence_len,padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [24]:
## Create predictors and label
import tensorflow as tf
x,y = input_sequences[:,:-1] , input_sequences[:,-1]

In [25]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32)

In [26]:
y

array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32)

In [27]:
y = tf.keras.utils.to_categorical(y,num_classes= total_words)
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [28]:
# split the data into training and testing sets
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
# Train out LSTM RNN